Exercise 7: NER

Part A:

The following code is copied from the lecture notebook:

In [1]:
import datasets
import evaluate
import transformers
import torch
from pprint import pprint    # pretty-print
import tabulate
from sklearn import metrics

In [2]:
dataset = datasets.load_dataset("dany0407/conll2003")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [3]:
POS_TAG_NAMES = dataset['train'].features['pos_tags'].feature.names
NER_TAG_NAMES = dataset['train'].features['ner_tags'].feature.names
CHUNK_TAG_NAMES = dataset['train'].features['chunk_tags'].feature.names

In [4]:
POS2ID = { n: i for i, n in enumerate(POS_TAG_NAMES) }
ID2POS = { i: n for i, n in enumerate(POS_TAG_NAMES) }

NER2ID = { n: i for i, n in enumerate(NER_TAG_NAMES) }
ID2NER = { i: n for i, n in enumerate(NER_TAG_NAMES) }

CHUNK2ID = { n: i for i, n in enumerate(CHUNK_TAG_NAMES) }
ID2CHUNK = { i: n for i, n in enumerate(CHUNK_TAG_NAMES) }

In [5]:
# From the documentation page and from here https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html

POS2DESCRIPTION = {
    "CC": "Coordinating conjunction",
    "CD": "Cardinal number",
    "DT": "Determiner",
    "EX": "Existential there",
    "FW": "Foreign word",
    "IN": "Preposition or subordinating conjunction",
    "JJ": "Adjective",
    "JJR": "Adjective, comparative",
    "JJS": "Adjective, superlative",
    "LS": "List item marker",
    "MD": "Modal",
    "NN": "Noun, singular or mass",
    "NNS": "Noun, plural",
    "NNP": "Proper noun, singular",
    "NNPS": "Proper noun, plural",
    "PDT": "Predeterminer",
    "POS": "Possessive ending",
    "PRP": "Personal pronoun",
    "PRP$": "Possessive pronoun",
    "RB": "Adverb",
    "RBR": "Adverb, comparative",
    "RBS": "Adverb, superlative",
    "RP": "Particle",
    "SYM": "Symbol",
    "TO": "to",
    "UH": "Interjection",
    "VB": "Verb, base form",
    "VBD": "Verb, past tense",
    "VBG": "Verb, gerund or present participle",
    "VBN": "Verb, past participle",
    "VBP": "Verb, non-3rd person singular present",
    "VBZ": "Verb, 3rd person singular present",
    "WDT": "Wh-determiner",
    "WP": "Wh-pronoun",
    "WP$": "Possessive wh-pronoun",
    "WRB": "Wh-adverb"
}

In [6]:
e = dataset["train"][12]    # work on the same example

table = []
for token, pos_id, chunk_id, ner_id in zip(e["tokens"], e["pos_tags"], e["chunk_tags"], e["ner_tags"]):
    ner_tag = ID2NER[ner_id]
    chunk_tag = ID2CHUNK[chunk_id]
    pos_tag = ID2POS[pos_id]
    pos_def = POS2DESCRIPTION.get(pos_tag,pos_tag)
    table.append([token, ner_tag, chunk_tag, pos_tag, pos_def])

print(tabulate.tabulate(table,headers=["Token", "NER", "Chunk", "POS", "POS definition"]))

Token     NER    Chunk    POS    POS definition
--------  -----  -------  -----  ------------------------
Only      O      B-NP     RB     Adverb
France    B-LOC  I-NP     NNP    Proper noun, singular
and       O      I-NP     CC     Coordinating conjunction
Britain   B-LOC  I-NP     NNP    Proper noun, singular
backed    O      B-VP     VBD    Verb, past tense
Fischler  B-PER  B-NP     NNP    Proper noun, singular
's        O      B-NP     POS    Possessive ending
proposal  O      I-NP     NN     Noun, singular or mass
.         O      O        .      .


1) Every focus word gets ignored:

In [7]:
'''
def token_features(tokens, index, window_size):
    # Generate features for token in position `index` in given list of tokens
    features = []

    # Context window start and end
    window_start = max(0, index-window_size)
    window_end = min(index+window_size+1, len(tokens))    # note +1 for range

    for i in range(window_start, window_end):
          offset = i - index    # relative position
          if offset == 0:
               pass
          else:
            features.append(f"token[{offset}]={tokens[i]}")

    """
    # Example custom feature: does focus token start with an upper-case letter?
    if tokens[index][0].isupper():
        features.append("first-letter-capitalized")
    """

    return features
'''

'\ndef token_features(tokens, index, window_size):\n    # Generate features for token in position `index` in given list of tokens\n    features = []\n\n    # Context window start and end\n    window_start = max(0, index-window_size)\n    window_end = min(index+window_size+1, len(tokens))    # note +1 for range\n\n    for i in range(window_start, window_end):\n          offset = i - index    # relative position\n          if offset == 0:\n               pass\n          else:\n            features.append(f"token[{offset}]={tokens[i]}")\n\n    """\n    # Example custom feature: does focus token start with an upper-case letter?\n    if tokens[index][0].isupper():\n        features.append("first-letter-capitalized")\n    """\n\n    return features\n'

2) Everything else gets ignored but not the focus word

In [ ]:
'''
def token_features(tokens, index, window_size):
    # Generate features for token in position `index` in given list of tokens
    features = []

    # Context window start and end
    window_start = max(0, index-window_size)
    window_end = min(index+window_size+1, len(tokens))    # note +1 for range

    for i in range(window_start, window_end):
          offset = i - index    # relative position
          if offset == 0:
            features.append(f"token[{offset}]={tokens[i]}")
          else:
            pass

    """
    # Example custom feature: does focus token start with an upper-case letter?
    if tokens[index][0].isupper():
        features.append("first-letter-capitalized")
    """

    return features
'''

In [9]:
def add_features_to_sentence(sentence):
    # Collect lists of features for all tokens here
    all_features = []

    tokens = sentence["tokens"]
    for index in range(len(tokens)):
        all_features.append(" ".join(token_features(tokens, index, window_size=3)))

    return { "features": all_features}

In [10]:
for feats in add_features_to_sentence(dataset["train"][12])["features"]:
    print(feats)

token[0]=Only
token[0]=France
token[0]=and
token[0]=Britain
token[0]=backed
token[0]=Fischler
token[0]='s
token[0]=proposal
token[0]=.


In [11]:
dataset = dataset.map(add_features_to_sentence)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [12]:
pprint(dataset["train"][12])

{'chunk_tags': [11, 12, 12, 12, 21, 11, 11, 12, 0],
 'features': ['token[0]=Only',
              'token[0]=France',
              'token[0]=and',
              'token[0]=Britain',
              'token[0]=backed',
              'token[0]=Fischler',
              "token[0]='s",
              'token[0]=proposal',
              'token[0]=.'],
 'id': '12',
 'ner_tags': [0, 5, 0, 5, 0, 1, 0, 0, 0],
 'pos_tags': [30, 22, 10, 22, 38, 22, 27, 21, 7],
 'tokens': ['Only',
            'France',
            'and',
            'Britain',
            'backed',
            'Fischler',
            "'s",
            'proposal',
            '.']}


In [13]:
def flatten(subset):
    # Keys for values to flatten
    keys = ["tokens", "pos_tags", "chunk_tags", "ner_tags", "features"]

    # Initialize to empty lists of tokens etc.
    flattened = { k: [] for k in keys }

    # Concatenate per-sentence lists of tokens etc.
    for sentence in subset:
        for key in keys:
            flattened[key].extend(sentence[key])

    # Return as Dataset object
    return datasets.Dataset.from_dict(flattened)

In [14]:
flattened_dict = {
    "train": flatten(dataset["train"]),
    "validation": flatten(dataset["validation"]),
    "test": flatten(dataset["test"]),
}

flat_dataset = datasets.DatasetDict(flattened_dict)

In [15]:
flat_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'features'],
        num_rows: 203621
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'features'],
        num_rows: 51362
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'features'],
        num_rows: 46435
    })
})

In [16]:
for i in range(10):
    token = flat_dataset["train"]["tokens"][i]
    pos_tag = ID2POS[flat_dataset["train"]["pos_tags"][i]]
    description = POS2DESCRIPTION.get(pos_tag, pos_tag)
    features = flat_dataset["train"]["features"][i]
    print(f"{token}\t{pos_tag}\t{description}\t{features}")

EU	NNP	Proper noun, singular	token[0]=EU
rejects	VBZ	Verb, 3rd person singular present	token[0]=rejects
German	JJ	Adjective	token[0]=German
call	NN	Noun, singular or mass	token[0]=call
to	TO	to	token[0]=to
boycott	VB	Verb, base form	token[0]=boycott
British	JJ	Adjective	token[0]=British
lamb	NN	Noun, singular or mass	token[0]=lamb
.	.	.	token[0]=.
Peter	NNP	Proper noun, singular	token[0]=Peter


In [17]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import WhitespaceSplit
from transformers import PreTrainedTokenizerFast

def build_whitespace_tokenizer(dataset, vocab_size=None, min_frequency=2):
    tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))

    # Use pure whitespace splitting, no normalization or subwording
    tokenizer.normalizer = None #This is a step which happens before tokenization, we want nothing done. So setting to None.
    tokenizer.pre_tokenizer = WhitespaceSplit() #This is a pre-tokenization step, we want to strictly only use whitespace.

    # List [PAD] first, so it gets index 0
    special_tokens = ["[PAD]","[UNK]"]

    # WordLevelTrainer builds a word-level tokenizer, i.e. no subwords are used, no BPE algorithm, simply full words
    # Words that do not fit into the vocabulary will be replaced by [UNK], which is fine for our purposes
    trainer = WordLevelTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency, #only consider tokens which appear this many times
        special_tokens=special_tokens,
    )
    # This is where the actual training happens
    iterator = (example["features"] for example in dataset)
    tokenizer.train_from_iterator(iterator, trainer=trainer)
    print(tokenizer)

    # And this then turns the tokenizer into something we can use
    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,

        unk_token="[UNK]",
        pad_token="[PAD]",
    )

    return hf_tokenizer
tokenizer = build_whitespace_tokenizer(flat_dataset["train"], vocab_size=15000)




Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"[PAD]", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":1, "content":"[UNK]", "single_word":False, "lstrip":False, "rstrip":False, ...}], normalizer=None, pre_tokenizer=WhitespaceSplit(), post_processor=None, decoder=None, model=WordLevel(vocab={"[PAD]":0, "[UNK]":1, "token[0]=.":2, "token[0]=,":3, "token[0]=the":4, ...}, unk_token="[UNK]"))


In [18]:
tokenizer.vocab

{'token[0]=third-round': 11807,
 'token[0]=lands': 7686,
 'token[0]=thrown': 4887,
 'token[0]=discovery': 10860,
 'token[0]=brokerage': 5694,
 'token[0]=District': 2286,
 'token[0]=Geneva': 3686,
 'token[0]=focus': 5822,
 'token[0]=7,011': 8633,
 'token[0]=guided': 7611,
 'token[0]=as': 31,
 'token[0]=activity': 1720,
 'token[0]=union': 1216,
 'token[0]=Gurion': 9352,
 'token[0]=473-6': 6340,
 'token[0]=5-7': 1670,
 'token[0]=dead': 1089,
 'token[0]=mountainous': 7741,
 'token[0]=borders': 5687,
 'token[0]=line': 1044,
 'token[0]=problems': 1101,
 'token[0]=Jaffray': 9482,
 'token[0]=repeatedly': 3067,
 'token[0]=largest': 1449,
 'token[0]=Blanco': 8893,
 'token[0]=Ata-ur-Rehman': 4228,
 'token[0]=Troy': 5583,
 'token[0]=Three': 2146,
 'token[0]=Jornada': 6817,
 'token[0]=will': 45,
 'token[0]=lawsuits': 7687,
 'token[0]=Constitutional': 9044,
 'token[0]=1980s': 3561,
 'token[0]=immigration': 5866,
 'token[0]=attending': 3334,
 'token[0]=herbicides': 11067,
 'token[0]=absolute': 10498,

In [19]:
def encode(examples):
    return tokenizer(examples['features'])

dset_tokenized = flat_dataset.map(encode,batched=True,num_proc=4)

for key,val in dset_tokenized["train"][0].items():
    print(key,":",val)

Map (num_proc=4):   0%|          | 0/203621 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/51362 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/46435 [00:00<?, ? examples/s]

tokens : EU
pos_tags : 22
chunk_tags : 11
ner_tags : 3
features : token[0]=EU
input_ids : [964]
attention_mask : [1]


In [20]:
def assign_labels(ex):
    return {"labels":ex["pos_tags"]}
dset_tokenized=dset_tokenized.map(assign_labels,batched=True)

Map:   0%|          | 0/203621 [00:00<?, ? examples/s]

Map:   0%|          | 0/51362 [00:00<?, ? examples/s]

Map:   0%|          | 0/46435 [00:00<?, ? examples/s]

In [21]:
dset_tokenized["train"][0]

{'tokens': 'EU',
 'pos_tags': 22,
 'chunk_tags': 11,
 'ner_tags': 3,
 'features': 'token[0]=EU',
 'input_ids': [964],
 'attention_mask': [1],
 'labels': 22}

In [22]:
# A Transformers library model wants a config,
# I can simply inherit from the base
# class for pretrained configs
# We don't really need to do anything very special
class MLPConfig(transformers.PretrainedConfig):
    pass

# This is the model
class MLP(transformers.PreTrainedModel):

    config_class=MLPConfig

    # In the initialization method, one instantiates the layers
    # these will be, for the most part the trained parameters of the model
    def __init__(self,config):
        super().__init__(config)
        self.all_tied_weights_keys = {} #Annoying bug in Transformers, must have this here or else we crash on saved model load
        #### HERE WE CREATE THE MODEL'S LAYERS:
        self.vocab_size=config.vocab_size #embedding matrix row count
        # Build and initialize embedding of vocab size x hidden size
        assert tokenizer.vocab["[PAD]"]==0 #let's make sure our assumption of pad==0 holds!

        self.embedding=torch.nn.Embedding(num_embeddings=self.vocab_size,embedding_dim=config.hidden_size,padding_idx=0)
        # Initialize the embeddings to random values
        # Note! This function is relatively clever and keeps the embedding for 0, the padding, pure zeros
        torch.nn.init.uniform_(self.embedding.weight.data,-0.001,0.001) #initialize the embeddings with small random values

        # This takes care of the lower half of the network, now the upper half
        # Output layer: hidden size x output size
        self.output=torch.nn.Linear(in_features=config.hidden_size,out_features=config.nlabels)
        # Now we have the parameters of the model
        self.loss=torch.nn.CrossEntropyLoss() #This loss is meant for classification, so let's use it


    # The computation of the model is put into the forward() function
    # it receives a batch of data and optionally the correct `labels`
    #
    # If given `labels` it returns (loss,output)
    # if not, then it returns (output,)
    def forward(self,input_ids,labels=None,**kwargs):
        #1) sum up the embeddings of the items
        embedded=self.embedding(input_ids) #(batch,ids)->(batch,ids,embedding_dim)
        # Since the Embedding keeps the first row of the matrix pure zeros, we don't need to worry about the padding
        # so next we sum the embeddings across the word dimension
        # (batch,ids,embedding_dim) -> (batch,embedding_dim)
        embedded_summed=torch.sum(embedded,dim=1)

        #2) apply non-linearity
        # (batch,embedding_dim) -> (batch,embedding_dim)
        projected=torch.tanh(embedded_summed) #Note how non-linearity is applied here and not when configuring the layer in __init__()
        # projected=embedded_summed ## If you need to bypass non-linearity, use this :)
        #3) and now apply the upper, output layer of the network
        # (batch,embedding_dim) -> (batch, num_of_classes i.e. 2 in our case)
        logits=self.output(projected)

        # ...and that's all there is to it!

        #print("input_ids.shape",input_ids.shape)
        #print("embedded.shape",embedded.shape)
        #print("embedded_summed.shape",embedded_summed.shape)
        #print("projected.shape",projected.shape)
        #print("logits.shape",logits.shape)

        # If we have labels, we ought to calculate the loss
        if labels is not None:

            # You run it as loss(model_output,correct_labels)
            return (self.loss(logits,labels),logits) #2-tuple, i.e. pair of values returned
        else:
            # No labels, so just return the logits
            return (logits,) #this weird syntax means 1-tuple

In [23]:
num_labels = len(POS2ID)

mlp_config = MLPConfig(
    vocab_size=len(tokenizer.vocab),
    hidden_size=20,
    nlabels=num_labels
)

print("mlp_config:", mlp_config)

mlp_config: MLPConfig {
  "hidden_size": 20,
  "nlabels": 47,
  "transformers_version": "5.3.0",
  "vocab_size": 11984
}



In [24]:
trainer_args = transformers.TrainingArguments(
    "mlp_checkpoints", #save checkpoints here
    eval_strategy="steps",
    logging_strategy="steps",
    eval_steps=1000,
    logging_steps=1000,
    save_steps=1000,
    learning_rate=5e-4, #learning rate of the gradient descent
    max_steps=20000,
    load_best_model_at_end=True,
    per_device_train_batch_size=16
)

pprint(trainer_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1000,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=Fals

In [25]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = np.argmax(outputs, axis=-1) #pick the index of the "winning" label
    return accuracy.compute(predictions=predictions, references=labels)

In [26]:
# Make a new model
mlp = MLP(mlp_config)


# Argument gives the number of steps of patience before early stopping
# i.e. training is stopped when the evaluation loss fails to improve
# certain number of times
early_stopping = transformers.EarlyStoppingCallback(5)

trainer = transformers.Trainer(
    model=mlp,
    args=trainer_args,
    train_dataset=dset_tokenized["train"],
    eval_dataset=dset_tokenized["validation"].select(range(1000)),
    compute_metrics=compute_accuracy,
    data_collator=transformers.DataCollatorWithPadding(tokenizer),
    callbacks=[early_stopping]
)

trainer.train()

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
1000,3.304097,2.563615,0.619000
2000,2.152066,1.799803,0.716000
3000,1.632205,1.407089,0.762000
4000,1.300998,1.159720,0.790000
5000,1.107335,0.995736,0.793000
6000,0.990463,0.892423,0.795000
7000,0.861865,0.810182,0.808000
8000,0.790795,0.754395,0.815000
9000,0.749597,0.711743,0.820000
10000,0.696876,0.675929,0.824000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20000, training_loss=0.9621221801757812, metrics={'train_runtime': 16.6483, 'train_samples_per_second': 19221.184, 'train_steps_per_second': 1201.324, 'total_flos': 1894974858.0, 'train_loss': 0.9621221801757812, 'epoch': 1.571462245619549})

#### With ignoring the focus word the accuracy dropped to around 50 %, and when only focus words where shown the accuracy stated around the same so 84%

Part B

In this part I only changed the 'assign_labels' function to pick the ner tags instead of pos. Additionally, I added the pos tags to features, since this was recommended in the task. Though I assume, it wouldn't affect the outcome of the tagger with or without it. 

In [ ]:
def token_features(tokens, pos_tags, index, window_size):
    # Generate features for token in position `index` in given list of tokens
    features = []

    # Context window start and end
    window_start = max(0, index-window_size)
    window_end = min(index+window_size+1, len(tokens))    # note +1 for range

    for i in range(window_start, window_end):
        offset = i - index    # relative position
        features.append(f"token[{offset}]={tokens[i]}")
        features.append(f"pos_tag[{offset}]={ID2POS[pos_tags[i]]}") ## THIS ADDED

    # Example custom feature: does focus token start with an upper-case letter?
    if tokens[index][0].isupper():
        features.append("first-letter-capitalized")

    return features

In [ ]:
def add_features_to_sentence(sentence):
    # Collect lists of features for all tokens here
    all_features = []

    tokens = sentence["tokens"]
    pos_tags = sentence["pos_tags"] ## THIS ADDED
    for index in range(len(tokens)):
        all_features.append(" ".join(token_features(tokens, pos_tags, index, window_size=3)))

    return { "features": all_features}


In [ ]:
def assign_labels(ex):
    return {"labels":ex["ner_tags"]} ## THIS CHANGED
dset_tokenized=dset_tokenized.map(assign_labels,batched=True)

Map:   0%|          | 0/203621 [00:00<?, ? examples/s]

Map:   0%|          | 0/51362 [00:00<?, ? examples/s]

Map:   0%|          | 0/46435 [00:00<?, ? examples/s]

In [30]:
num_labels = len(NER2ID)

mlp_config = MLPConfig(
    vocab_size=len(tokenizer.vocab),
    hidden_size=20,
    nlabels=num_labels
)

print("mlp_config:", mlp_config)

mlp_config: MLPConfig {
  "hidden_size": 20,
  "nlabels": 9,
  "transformers_version": "5.3.0",
  "vocab_size": 11984
}



In [31]:
trainer_args = transformers.TrainingArguments(
    "NER_mlp_checkpoints", #save checkpoints here
    eval_strategy="steps",
    logging_strategy="steps",
    eval_steps=1000,
    logging_steps=1000,
    save_steps=1000,
    learning_rate=5e-4, #learning rate of the gradient descent
    max_steps=20000,
    load_best_model_at_end=True,
    per_device_train_batch_size=16
)

pprint(trainer_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1000,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=Fals

In [32]:
# Make a new model
NERmlp = MLP(mlp_config)


# Argument gives the number of steps of patience before early stopping
# i.e. training is stopped when the evaluation loss fails to improve
# certain number of times
early_stopping = transformers.EarlyStoppingCallback(5)

trainer = transformers.Trainer(
    model=NERmlp,
    args=trainer_args,
    train_dataset=dset_tokenized["train"],
    eval_dataset=dset_tokenized["validation"].select(range(1000)),
    compute_metrics=compute_accuracy,
    data_collator=transformers.DataCollatorWithPadding(tokenizer),
    callbacks=[early_stopping]
)

trainer.train()

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
1000,1.560979,1.062598,0.804000
2000,0.826936,0.783933,0.803000
3000,0.640985,0.671563,0.817000
4000,0.534659,0.605491,0.832000
5000,0.465647,0.560258,0.848000
6000,0.445248,0.525877,0.865000
7000,0.393433,0.498551,0.886000
8000,0.393043,0.476004,0.893000
9000,0.369741,0.458764,0.898000
10000,0.349103,0.446380,0.899000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20000, training_loss=0.4510383880615234, metrics={'train_runtime': 16.8331, 'train_samples_per_second': 19010.167, 'train_steps_per_second': 1188.135, 'total_flos': 362867526.0, 'train_loss': 0.4510383880615234, 'epoch': 1.571462245619549})

In [35]:
weights=mlp.embedding.weight.detach().cpu().numpy()

In [36]:
qry_idx=tokenizer.vocab["token[0]=in"]
idx2feat={v:k for k,v in tokenizer.vocab.items()}

#calculate the distance of the "in" embedding to all other embeddings
distance_to_qry=metrics.pairwise.euclidean_distances(weights[qry_idx:qry_idx+1,:],weights)
nearest_neighbors=np.argsort(distance_to_qry) #indices of words nearest to "in"
for nearest in nearest_neighbors[0,:20]:
    print(idx2feat[nearest])

token[0]=in
token[0]=on
token[0]=after
token[0]=In
token[0]=under
token[0]=into
token[0]=than
token[0]=between
token[0]=since
token[0]=if
token[0]=At
token[0]=during
token[0]=through
token[0]=before
token[0]=by
token[0]=because
token[0]=from
token[0]=behind
token[0]=OF
token[0]=with


In [37]:
NERweights=NERmlp.embedding.weight.detach().cpu().numpy()

In [38]:
qry_idx=tokenizer.vocab["token[0]=in"]
idx2feat={v:k for k,v in tokenizer.vocab.items()}

#calculate the distance of the "in" embedding to all other embeddings
distance_to_qry=metrics.pairwise.euclidean_distances(NERweights[qry_idx:qry_idx+1,:],NERweights)
nearest_neighbors=np.argsort(distance_to_qry) #indices of words nearest to "in"
for nearest in nearest_neighbors[0,:20]:
    print(idx2feat[nearest])

token[0]=in
token[0]=on
token[0]=1
token[0]=3
token[0]=told
token[0]=1996-08-28
token[0]=under
token[0]=10
token[0]=some
token[0]=time
token[0]=new
token[0]=about
token[0]=we
token[0]=Monday
token[0]=9
token[0]=Friday
token[0]=people
token[0]=more
token[0]=market
token[0]=or


#### When comparing the differences in the only pos and the ner+pos weights, the ner+pos has a bigger variety of words shown because I assume, the token 'in' is probably categorized as Date and Time, which affects the grouping that is seen in the NERweights.